In [1]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "-1"
import numpy as np
import tensorflow as tf
import librosa
import pickle
import soundfile as sf
import matplotlib.pyplot as plt
from tqdm import tqdm
import scipy.signal as signal

2025-05-18 00:44:50.699250: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-05-18 00:44:50.859664: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1747500290.924876 1652661 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1747500290.942905 1652661 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1747500291.083289 1652661 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking 

In [11]:
def generate_synthetic_audio(emotion, duration=2.5, sr=22050, data=None, saliency_data=None):
    """
    Generate synthetic audio with characteristics matching a specific emotion,
    incorporating saliency curves to focus emotional characteristics where the model pays most attention
    """
    if data is None:
        data = create_emotion_data_dict()
    
    if saliency_data is None:
        # Create default saliency curve (bell-shaped centered in the middle)
        time = np.linspace(0, duration, 100)
        kde_scaled = np.exp(-((time - duration/2) ** 2) / 0.5)
        saliency = {
            'time': time,
            'kde_scaled': kde_scaled / np.max(kde_scaled)  # Normalize to [0, 1]
        }
    else:
        # Use the provided saliency data for the specific emotion
        if emotion in saliency_data:
            saliency = saliency_data[emotion]
        else:
            # Default if the specific emotion saliency is not available
            time = np.linspace(0, duration, 100)
            kde_scaled = np.exp(-((time - duration/2) ** 2) / 0.5)
            saliency = {
                'time': time,
                'kde_scaled': kde_scaled / np.max(kde_scaled)  # Normalize to [0, 1]
            }
    
    # Get the emotion data
    emotion_data = data[emotion]
    
    # Create a base audio signal
    t = np.arange(0, duration, 1/sr)
    
    # Create a saliency curve that spans the full audio duration
    # Interpolate saliency values to match the audio length
    saliency_curve = np.interp(t, saliency['time'], saliency['kde_scaled'])
    
    # We'll use a mix of frequencies to create a synthetic speech-like signal
    # Start with vocal tract resonance frequencies (formants) for basic speech-like sound
    f1 = 500  # First formant
    f2 = 1500  # Second formant
    f3 = 2500  # Third formant
    
    # Create a base signal with formants
    audio = 0.1 * np.sin(2 * np.pi * f1 * t)
    audio += 0.05 * np.sin(2 * np.pi * f2 * t)
    audio += 0.025 * np.sin(2 * np.pi * f3 * t)
    
    # Add emotion-specific variation
    # Add some randomized harmonics based on emotion
    harmonics = []
    if emotion == 'ANGRY':
        # More high-frequency content for anger
        harmonics = [1000, 2000, 3000, 4000]
        harmonic_amps = [0.09, 0.07, 0.05, 0.03]
    elif emotion == 'SAD':
        # Less high-frequency, more low-frequency for sadness
        harmonics = [300, 600, 900, 1200]
        harmonic_amps = [0.09, 0.06, 0.04, 0.02]
    elif emotion == 'HAPPY':
        # Balanced spectrum with emphasis on mid frequencies
        harmonics = [800, 1600, 2400, 3200]
        harmonic_amps = [0.08, 0.06, 0.05, 0.04]
    elif emotion == 'FEAR':
        # Irregular spectrum with some sharp peaks
        harmonics = [600, 1200, 2000, 3500]
        harmonic_amps = [0.08, 0.07, 0.04, 0.06]
    elif emotion == 'DISGUST':
        # Rougher spectrum
        harmonics = [700, 1400, 2100, 3800]
        harmonic_amps = [0.07, 0.05, 0.06, 0.04]
    else:  # NEUTRAL
        # More balanced spectrum
        harmonics = [800, 1600, 2400, 3200]
        harmonic_amps = [0.06, 0.05, 0.04, 0.03]
    
    # Add the harmonics to the base signal, modulated by the saliency curve
    for freq, amp in zip(harmonics, harmonic_amps):
        # Modulate the amplitude based on the saliency curve
        modulated_amp = amp * saliency_curve
        audio += modulated_amp[:len(audio)] * np.sin(2 * np.pi * freq * t)
    
    # Add some random fluctuations to simulate voice variations
    # Scale fluctuation by saliency curve to focus noise in important areas
    fluctuation = np.random.normal(0, 0.01, len(audio)) * saliency_curve
    audio += fluctuation
    
    # Apply amplitude modulation based on RMSE
    rmse_mean = emotion_data['RMSE_mean']
    rmse_std = emotion_data['RMSE_std']
    
    # Create frames for RMSE modulation
    frame_length = 0.025  # 25ms frames
    hop_length = 0.010   # 10ms hop
    n_frames = int((duration - frame_length) / hop_length) + 1
    
    # Create RMSE envelope with correct statistical properties
    rmse_envelope = np.random.normal(rmse_mean, rmse_std, n_frames)
    rmse_envelope = np.abs(rmse_envelope)  # Ensure positive values
    
    # Stretch RMSE envelope to audio length and modulate by saliency
    frame_times = np.linspace(0, duration, n_frames)
    rmse_curve = np.interp(t, frame_times, rmse_envelope)
    
    # Apply saliency-weighted RMSE modulation
    weighted_rmse = rmse_curve * (0.3 + 0.7 * saliency_curve)  # Keep baseline plus saliency contribution
    audio *= weighted_rmse / np.mean(weighted_rmse)  # Normalize
    
    # Add noise with ZCR characteristics that follows saliency curve
    zcr_factor = emotion_data['ZCR_mean'] * 10  # Scale factor for noise amplitude
    noise_base = np.random.normal(0, zcr_factor, len(audio))
    
    # Shape noise by saliency curve
    shaped_noise = noise_base * (0.2 + 0.8 * saliency_curve)  # Add some baseline noise everywhere
    audio += shaped_noise
    
    # Apply some filtering based on MFCCs
    # We'll use a simple filter bank approach
    mfcc_means = emotion_data['MFCC_means']
    
    # Create different frequency components based on the first few MFCCs
    mfcc_contribution = np.zeros_like(audio)
    for i in range(min(6, len(mfcc_means))):  # Use first 6 MFCCs for main shaping
        freq = 300 + i * 250  # Simple frequency spacing
        amp = np.abs(mfcc_means[i] / 500)  # Scale MFCC to reasonable amplitude
        mfcc_contribution += amp * np.sin(2 * np.pi * freq * t)
    
    # Apply MFCC contribution modulated by saliency
    audio += mfcc_contribution * saliency_curve * 0.15
    
    # Apply vocal characteristics based on emotion, modulated by saliency
    if emotion == 'ANGRY':
        # Add more high frequency content and roughness where saliency is high
        roughness = np.random.normal(0, 0.2, len(audio)) * saliency_curve
        audio += roughness
        
        # Add trembling effect for anger, stronger at high saliency points
        tremolo_freq = 12  # Hz
        tremolo_depth = 0.2 * saliency_curve
        tremolo = 1 + tremolo_depth * np.sin(2 * np.pi * tremolo_freq * t)
        audio *= tremolo
        
        # Add bursts of energy at peak saliency points
        peak_indices = signal.find_peaks(saliency_curve, height=0.7)[0]
        for idx in peak_indices:
            if idx < len(audio):
                # Create a short burst window
                burst_width = int(0.05 * sr)  # 50ms burst
                burst_start = max(0, idx - burst_width//2)
                burst_end = min(len(audio), idx + burst_width//2)
                burst_window = np.hanning(burst_end - burst_start)
                audio[burst_start:burst_end] *= (1 + 0.5 * burst_window)
        
    elif emotion == 'SAD':
        # Add slower modulation for sadness
        mod_freq = 3  # Hz
        mod_depth = 0.1 * saliency_curve
        slow_mod = 1 + mod_depth * np.sin(2 * np.pi * mod_freq * t)
        audio *= slow_mod
        
        # Filter to reduce high frequencies more at important parts
        # Create a time-varying filter cutoff based on saliency
        cutoffs = 0.3 - 0.15 * saliency_curve  # Lower cutoff at high saliency
        
        # Apply time-varying filter (simplified approach)
        filtered_audio = np.zeros_like(audio)
        for i in range(0, len(audio), sr//10):  # Process in 100ms chunks
            end = min(i + sr//10, len(audio))
            segment = audio[i:end]
            if len(segment) > 0:
                cutoff = cutoffs[i] if i < len(cutoffs) else 0.3
                b, a = signal.butter(3, cutoff, 'low')
                filtered_segment = signal.lfilter(b, a, segment)
                filtered_audio[i:end] = filtered_segment
        
        audio = filtered_audio
        
    elif emotion == 'HAPPY':
        # Add faster modulation for happiness
        mod_freq = 8  # Hz
        mod_depth = 0.15 * saliency_curve
        fast_mod = 1 + mod_depth * np.sin(2 * np.pi * mod_freq * t)
        audio *= fast_mod
        
        # Boost mid frequencies more at important parts
        # Create a time-varying filter band based on saliency
        low_cutoffs = 0.1 * np.ones_like(saliency_curve)
        high_cutoffs = 0.7 + 0.2 * saliency_curve  # Wider band at important points
        
        # Apply time-varying filter (simplified approach)
        filtered_audio = np.zeros_like(audio)
        for i in range(0, len(audio), sr//10):  # Process in 100ms chunks
            end = min(i + sr//10, len(audio))
            segment = audio[i:end]
            if len(segment) > 0:
                low_cut = low_cutoffs[i] if i < len(low_cutoffs) else 0.1
                high_cut = high_cutoffs[i] if i < len(high_cutoffs) else 0.7
                b, a = signal.butter(3, [low_cut, high_cut], 'band')
                filtered_segment = signal.lfilter(b, a, segment)
                filtered_audio[i:end] = filtered_segment
                
        audio = filtered_audio
        
    elif emotion == 'FEAR':
        # Add irregular trembling modulated by saliency
        base_freq = 5  # Hz
        var_freq = 0.5  # Hz
        trem_depth = 0.25 * saliency_curve
        irreg_trem = 1 + trem_depth * np.sin(2 * np.pi * (base_freq + np.sin(2 * np.pi * var_freq * t)) * t)
        audio *= irreg_trem
        
        # Add some breathiness, stronger at high saliency
        breathiness = np.random.normal(0, 0.15, len(audio)) * saliency_curve
        audio += breathiness
        
        # Add occasional silence gaps at high saliency points to simulate fearful pauses
        high_sal_points = signal.find_peaks(saliency_curve, height=0.8)[0]
        for idx in high_sal_points:
            if idx < len(audio):
                gap_width = int(0.03 * sr)  # 30ms gap
                gap_start = max(0, idx - gap_width//2)
                gap_end = min(len(audio), idx + gap_width//2)
                fade = np.hanning(gap_end - gap_start)
                audio[gap_start:gap_end] *= (1 - 0.9 * fade)  # Almost silence
        
    elif emotion == 'DISGUST':
        # Add roughness modulated by saliency
        roughness = np.random.normal(0, 0.15, len(audio)) * saliency_curve
        audio += roughness
        
        # Add distortion that increases with saliency
        distortion_factor = 1.5 * saliency_curve
        for i in range(len(audio)):
            if i < len(distortion_factor):
                audio[i] = np.tanh(distortion_factor[i] * audio[i])
            else:
                audio[i] = np.tanh(1.5 * audio[i])  # Default distortion
                
        # Add characteristic frequency dips at high saliency points
        high_sal_points = signal.find_peaks(saliency_curve, height=0.7)[0]
        for idx in high_sal_points:
            if idx < len(audio):
                dip_width = int(0.04 * sr)  # 40ms dip
                dip_start = max(0, idx - dip_width//2)
                dip_end = min(len(audio), idx + dip_width//2)
                if dip_end > dip_start:
                    # Apply a notch filter at this point
                    b, a = signal.iirnotch(1500, 30, sr)
                    audio[dip_start:dip_end] = signal.lfilter(b, a, audio[dip_start:dip_end])
    
#     # Apply emotion-specific amplitude scaling
#     if emotion == 'ANGRY':
#         # Make angry louder with more variations based on saliency
#         amplitude_factor = 1.5 * (0.8 + 0.4 * saliency_curve)
#         audio *= amplitude_factor
#     elif emotion == 'SAD':
#         # Make saimport numpy as np
# # import librosa
# import soundfile as sf
# import pickle
# import tensorflow as tf
# from scipy import signal

# Define the dictionary structure based on the provided data
def create_emotion_data_dict():
    # Use the provided emotion data structure directly
    data = {
        'ANGRY': {
            'ZCR_mean': 0.0846,
            'ZCR_std': 0.0296,
            'RMSE_mean': 0.0692,
            'RMSE_std': 0.0478,
            'MFCC_means': [
                -298.124, 121.593, -7.995, 41.206, -17.941, 
                9.999, -15.744, 4.588, -15.002, 1.481,
                -3.137, -5.640, 3.305, -9.326, 2.893,
                -9.847, 0.812, -8.127, 0.116, -3.868
            ],
            'MFCC_stds': [
                57.716, 17.098, 15.712, 11.359, 9.764,
                10.442, 7.793, 5.566, 4.787, 4.461,
                4.062, 3.706, 3.975, 3.546, 3.853,
                4.285, 4.058, 3.810, 3.839, 3.803
            ]
        },
        'DISGUST': {
            'ZCR_mean': 0.0708,
            'ZCR_std': 0.0300,
            'RMSE_mean': 0.0212,
            'RMSE_std': 0.0174,
            'MFCC_means': [
                -385.598, 137.840, 1.608, 51.619, -15.655,
                22.205, -18.391, 9.706, -13.179, 3.508,
                -0.987, -4.419, 4.442, -9.774, 5.044,
                -10.782, 2.495, -8.586, 0.965, -4.649
            ],
            'MFCC_stds': [
                46.325, 14.257, 11.958, 12.195, 7.744,
                10.923, 6.812, 6.224, 4.110, 4.033,
                3.871, 3.139, 3.298, 2.954, 3.260,
                3.681, 3.200, 2.723, 2.941, 2.603
            ]
        },
        'FEAR': {
            'ZCR_mean': 0.0645,
            'ZCR_std': 0.0249,
            'RMSE_mean': 0.0303,
            'RMSE_std': 0.0310,
            'MFCC_means': [
                -379.109, 132.062, 4.086, 50.065, -14.141,
                21.857, -16.556, 8.178, -12.486, 2.971,
                -2.549, -5.420, 3.570, -9.721, 4.359,
                -10.538, 1.967, -8.359, 0.623, -4.244
            ],
            'MFCC_stds': [
                59.114, 15.984, 12.690, 12.302, 7.665,
                11.356, 6.379, 6.280, 3.946, 4.018,
                3.989, 3.531, 3.845, 3.502, 3.727,
                4.320, 3.925, 3.728, 3.632, 3.724
            ]
        },
        'HAPPY': {
            'ZCR_mean': 0.0671,
            'ZCR_std': 0.0255,
            'RMSE_mean': 0.0340,
            'RMSE_std': 0.0235,
            'MFCC_means': [
                -356.270, 133.676, 0.348, 45.186, -13.289,
                15.806, -16.239, 6.152, -13.946, 2.110,
                -2.470, -5.475, 3.416, -9.233, 4.003,
                -9.829, 1.275, -7.657, 0.534, -3.884
            ],
            'MFCC_stds': [
                47.268, 14.645, 13.520, 10.231, 8.411,
                10.349, 7.024, 5.542, 4.562, 4.244,
                4.177, 3.572, 3.609, 3.512, 3.479,
                4.405, 3.814, 3.650, 3.467, 3.484
            ]
        },
        'NEUTRAL': {
            'ZCR_mean': 0.0619,
            'ZCR_std': 0.0244,
            'RMSE_mean': 0.0167,
            'RMSE_std': 0.0062,
            'MFCC_means': [
                -398.677, 141.591, 6.849, 53.914, -14.457,
                21.870, -17.550, 8.230, -13.621, 3.048,
                -2.019, -4.408, 4.477, -9.200, 5.233,
                -10.508, 2.698, -8.454, 1.005, -4.425
            ],
            'MFCC_stds': [
                26.359, 12.097, 10.285, 8.730, 7.124,
                8.421, 6.028, 4.935, 3.831, 3.745,
                3.801, 2.985, 3.127, 2.819, 2.770,
                3.791, 2.989, 2.704, 2.824, 2.381
            ]
        },
        'SAD': {
            'ZCR_mean': 0.0553,
            'ZCR_std': 0.0218,
            'RMSE_mean': 0.0120,
            'RMSE_std': 0.0071,
            'MFCC_means': [
                -429.167, 143.558, 8.845, 58.525, -15.826,
                29.045, -17.954, 12.292, -12.491, 3.972,
                -1.606, -4.158, 4.409, -10.219, 5.771,
                -11.590, 3.432, -9.358, 1.490, -4.850
            ],
            'MFCC_stds': [
                37.168, 11.792, 9.642, 9.299, 6.888,
                9.068, 5.482, 5.340, 3.337, 3.396,
                3.323, 2.647, 2.934, 2.704, 2.720,
                3.494, 2.675, 2.329, 2.699, 2.500
            ]
        }
    }
    
    return data

def load_saliency_curves(kde_dir="KDE_vals"):
    """
    Load the saliency curves for each emotion
    """
    import os
    saliency_data = {}
    
    for emotion in ['ANGRY', 'DISGUST', 'FEAR', 'HAPPY', 'NEUTRAL', 'SAD']:
        try:
            sal_file = os.path.join(kde_dir, f"{emotion.lower()}.npz")
            if os.path.exists(sal_file):
                sal = np.load(sal_file)
                saliency_data[emotion] = {
                    'time': sal['time'],
                    'kde_scaled': sal['kde_scaled']
                }
            else:
                print(f"Warning: Saliency file not found for {emotion}")
                # Create default saliency curve (bell-shaped centered in the middle)
                time = np.linspace(0, 2.5, 100)
                kde_scaled = np.exp(-((time - 1.25) ** 2) / 0.5)
                saliency_data[emotion] = {
                    'time': time,
                    'kde_scaled': kde_scaled / np.max(kde_scaled)  # Normalize to [0, 1]
                }
        except Exception as e:
            print(f"Error loading saliency data for {emotion}: {e}")
            # Create default saliency curve
            time = np.linspace(0, 2.5, 100)
            kde_scaled = np.exp(-((time - 1.25) ** 2) / 0.5)
            saliency_data[emotion] = {
                'time': time,
                'kde_scaled': kde_scaled / np.max(kde_scaled)  # Normalize to [0, 1]
            }
    
    return saliency_data

def generate_synthetic_audio(emotion, duration=2.5, sr=22050, data=None):
    """
    Generate synthetic audio with characteristics matching a specific emotion
    """
    if data is None:
        data = create_emotion_data_dict()
    
    # Get the emotion data
    emotion_data = data[emotion]
    
    # Create a base audio signal
    t = np.arange(0, duration, 1/sr)
    
    # We'll use a mix of frequencies to create a synthetic speech-like signal
    # Start with vocal tract resonance frequencies (formants) for basic speech-like sound
    f1 = 500  # First formant
    f2 = 1500  # Second formant
    f3 = 2500  # Third formant
    
    # Create a base signal with formants
    audio = 0.1 * np.sin(2 * np.pi * f1 * t)
    audio += 0.05 * np.sin(2 * np.pi * f2 * t)
    audio += 0.025 * np.sin(2 * np.pi * f3 * t)
    
    # Add some random fluctuations to simulate voice variations
    fluctuation = np.random.normal(0, 0.01, len(audio))
    audio += fluctuation
    
    # Apply amplitude modulation based on RMSE
    rmse_mean = emotion_data['RMSE_mean']
    rmse_variation = np.random.normal(rmse_mean, emotion_data['RMSE_std'], len(t))
    rmse_variation = np.abs(rmse_variation)  # Ensure positive values
    
    # Create an envelope using the RMSE pattern
    envelope = np.interp(np.linspace(0, 1, len(audio)), np.linspace(0, 1, 100), 
                          np.random.normal(rmse_mean, emotion_data['RMSE_std'], 100))
    envelope = np.abs(envelope)
    
    # Apply the envelope
    audio *= envelope[:len(audio)]
    
    # Adjust overall amplitude
    if emotion == 'ANGRY':
        # Make angry louder
        audio *= 1.5
    elif emotion == 'SAD':
        # Make sad quieter
        audio *= 0.7
    
    # Add some noise with ZCR characteristics
    zcr_factor = emotion_data['ZCR_mean'] * 10  # Scale ZCR for noise addition
    noise = np.random.normal(0, zcr_factor, len(audio))
    audio += noise
    
    # Apply some filtering based on MFCCs
    # We'll use a simple filter bank approach
    mfcc_means = emotion_data['MFCC_means']
    
    # Create different frequency components based on the first few MFCCs
    for i in range(min(5, len(mfcc_means))):
        freq = 300 + i * 200  # Simple frequency spacing
        amp = np.abs(mfcc_means[i] / 500)  # Scale MFCC to reasonable amplitude
        audio += amp * np.sin(2 * np.pi * freq * t)
    
    # Apply vocal characteristics based on emotion
    if emotion == 'ANGRY':
        # Add more high frequency content and roughness
        roughness = np.random.normal(0, 0.2, len(audio))
        audio += roughness
        # Add trembling effect for anger
        tremolo = 1 + 0.2 * np.sin(2 * np.pi * 12 * t)
        audio *= tremolo
        
    elif emotion == 'SAD':
        # Add slower modulation for sadness
        slow_mod = 1 + 0.1 * np.sin(2 * np.pi * 3 * t)
        audio *= slow_mod
        # Filter to reduce high frequencies
        b, a = signal.butter(3, 0.3, 'low')
        audio = signal.lfilter(b, a, audio)
        
    elif emotion == 'HAPPY':
        # Add faster modulation for happiness
        fast_mod = 1 + 0.15 * np.sin(2 * np.pi * 8 * t)
        audio *= fast_mod
        # Boost mid frequencies
        b, a = signal.butter(3, [0.1, 0.7], 'band')
        audio = signal.lfilter(b, a, audio)
        
    elif emotion == 'FEAR':
        # Add irregular trembling
        irreg_trem = 1 + 0.25 * np.sin(2 * np.pi * (5 + np.sin(2 * np.pi * 0.5 * t)) * t)
        audio *= irreg_trem
        # Add some breathiness
        breathiness = np.random.normal(0, 0.15, len(audio))
        audio += breathiness
        
    elif emotion == 'DISGUST':
        # Add roughness
        roughness = np.random.normal(0, 0.15, len(audio))
        audio += roughness
        # Add some distortion
        audio = np.tanh(1.5 * audio)
        
    # Normalize
    audio = audio / np.max(np.abs(audio))
    
    return audio

def create_all_emotion_audio(output_dir="emotion_audio", sr=22050):
    """Create audio files for all emotions"""
    import os
    
    if not os.path.exists(output_dir):
        os.makedirs(output_dir)
    
    data = create_emotion_data_dict()
    emotions = list(data.keys())
    
    for emotion in emotions:
        print(f"Generating {emotion} audio...")
        audio = generate_synthetic_audio(emotion, duration=2.5, sr=sr, data=data)
        output_file = os.path.join(output_dir, f"{emotion.lower()}_synthetic.wav")
        sf.write(output_file, audio, sr)
        print(f"Saved to {output_file}")

def extract_features(data, sr=22050, frame_length=2048, hop_length=512):
    """
    Extract features (ZCR, RMSE, MFCC) from the audio data
    """
    # Zero Crossing Rate
    zcr = librosa.feature.zero_crossing_rate(data, frame_length=frame_length, hop_length=hop_length)
    
    # Root Mean Square Energy
    rmse = librosa.feature.rms(y=data, frame_length=frame_length, hop_length=hop_length)
    
    # MFCCs
    mfcc_features = librosa.feature.mfcc(y=data, sr=sr, n_mfcc=20, n_fft=frame_length, hop_length=hop_length)
    mfcc_features = mfcc_features.T  # Transpose to get time x features
    
    # Flatten all features for easier processing
    result = np.hstack((
        np.squeeze(zcr),
        np.squeeze(rmse),
        np.ravel(mfcc_features)
    ))
    
    return result

def load_model_and_predictors(model_json_path, model_weights_path, scaler_path, encoder_path):
    """
    Load the pre-trained model, scaler, and encoder
    """
    # Load model architecture
    with open(model_json_path, 'r') as json_file:
        loaded_model_json = json_file.read()
    loaded_model = tf.keras.models.model_from_json(loaded_model_json)
    
    # Load model weights
    loaded_model.load_weights(model_weights_path)
    
    # Compile model
    loaded_model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
    
    # Load scaler
    with open(scaler_path, "rb") as f:
        scaler = pickle.load(f)
    
    # Load encoder
    with open(encoder_path, "rb") as f:
        encoder = pickle.load(f)
    
    return loaded_model, scaler, encoder

def get_predict_feat(audio_data, sr, scaler, expected_shape=(1, 2376)):
    """
    Prepare features for prediction
    """
    res = extract_features(audio_data, sr)
    
    # Ensure res is reshaped or padded to match the expected shape
    if res.shape != expected_shape:
        flat_size = np.prod(expected_shape)
        if res.size < flat_size:
            # Pad if the size is smaller than expected
            pad_width = (0, flat_size - res.size)
            res = np.pad(res, pad_width=pad_width, mode='constant')
        else:
            # Resize if the size is larger than expected
            res = np.resize(res, expected_shape)

    i_result = scaler.transform(res.reshape(1, -1))
    final_result = np.expand_dims(i_result, axis=2)
    return final_result

def prediction(audio_data, sr, model, scaler, encoder):
    """
    Make a prediction using the loaded model
    """
    res = get_predict_feat(audio_data, sr, scaler)
    predictions = model.predict(res)
    
    # Get the label names
    label_names = list(encoder.categories_[0])
    
    # Get the index of the label with the highest confidence score
    predicted_label_index = np.argmax(predictions)
    
    # List to store confidence scores
    confidence_scores = []
    
    # Display predicted emotion and confidence for each label
    print(f"\nPredicted Emotion: {label_names[predicted_label_index]}")
    
    for label_index, label_name in enumerate(label_names):
        confidence_score = predictions[0][label_index]
        confidence_score = 0 if confidence_score < 0.001 else confidence_score
        confidence_scores.append({'label': label_name, 'confidence': confidence_score})
    
    sorted_confidence_scores = sorted(confidence_scores, key=lambda x: x['confidence'], reverse=True)
    return sorted_confidence_scores

def optimize_emotion_audio(emotion, model, scaler, encoder, iterations=20, sr=22050):
    """
    Optimize the audio generation to get high confidence for the target emotion
    """
    best_audio = None
    best_confidence = 0
    
    for i in range(iterations):
        print(f"Iteration {i+1}/{iterations}")
        
        # Generate audio with slightly different parameters each time
        audio = generate_synthetic_audio(emotion, duration=2.5, sr=sr)
        
        # Predict emotion
        results = prediction(audio, sr, model, scaler, encoder)
        
        # Check if this is the target emotion and has better confidence
        for result in results:
            if result['label'] == emotion:
                confidence = result['confidence']
                if confidence > best_confidence:
                    best_confidence = confidence
                    best_audio = audio
                    print(f"New best confidence for {emotion}: {best_confidence:.4f}")
                break
    
    if best_audio is not None:
        print(f"Best confidence achieved for {emotion}: {best_confidence:.4f}")
        return best_audio
    else:
        print(f"Could not optimize audio for {emotion}")
        # Return the last generated audio as fallback
        return audio

if __name__ == "__main__":
    # Example usage:
    # 1. Create audio files for all emotions
#     create_all_emotion_audio()
    
    # 2. If you want to optimize audio for a specific emotion with your model:
    
    # First load your model, scaler, and encoder
    model, scaler, encoder = load_model_and_predictors(
        "results/CNN_model.json",
        "results/best_model.weights.h5",
        "results/scaler.pickle",
        "results/encoder.pickle"
    )
    
    # Then optimize audio for a specific emotion
    emotion = "ANGRY"
    optimized_audio = optimize_emotion_audio(emotion, model, scaler, encoder)
    
    # Save the optimized audio
    sf.write(f"{emotion.lower()}_optimized.wav", optimized_audio, 22050)
    


/home/aegis/Research/Machine Learning in HEP/mlenv/lib/python3.12/site-packages/sklearn/base.py:376: InconsistentVersionWarning: Trying to unpickle estimator StandardScaler from version 1.2.2 when using version 1.5.2. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/home/aegis/Research/Machine Learning in HEP/mlenv/lib/python3.12/site-packages/sklearn/base.py:376: InconsistentVersionWarning: Trying to unpickle estimator OneHotEncoder from version 1.2.2 when using version 1.5.2. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


Iteration 1/20
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 189ms/step

Predicted Emotion: fear
Iteration 2/20
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 131ms/step

Predicted Emotion: angry
Iteration 3/20
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 93ms/step

Predicted Emotion: angry
Iteration 4/20
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 102ms/step

Predicted Emotion: angry
Iteration 5/20
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 98ms/step

Predicted Emotion: angry
Iteration 6/20
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 94ms/step

Predicted Emotion: angry
Iteration 7/20
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 104ms/step

Predicted Emotion: angry
Iteration 8/20
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 107ms/step

Predicted Emotion: angry
Iteration 9/20
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 103ms/step

Predicted Emotion: angry
Iteration 10/20
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 95ms/step

Predicted Emotion: angry
Iteration 11/20
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 88ms/step

Predicted Emotion: angry
Iteration 12/20
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 121ms/step

Predicted Emotion: angry
Iteration 13/20
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s

In [4]:
# Load the model (from your original code)
with open('results/CNN_model.json', 'r') as json_file:
    loaded_model_json = json_file.read()
loaded_model = tf.keras.models.model_from_json(loaded_model_json)
loaded_model.load_weights("results/best_model.weights.h5")
loaded_model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

# Load scaler & encoder
with open("results/scaler.pickle", "rb") as f:
    scaler = pickle.load(f)
with open("results/encoder.pickle", "rb") as f:
    encoder = pickle.load(f)

# Get label names
label_names = list(encoder.categories_[0])
print("Available emotions:", label_names)

# Feature extraction functions (from your original code)
def zcr(data, frame_length, hop_length):
    zcr = librosa.feature.zero_crossing_rate(data, frame_length=frame_length, hop_length=hop_length)
    return np.squeeze(zcr)

def rmse(data, frame_length=2048, hop_length=512):
    rmse = librosa.feature.rms(y=data, frame_length=frame_length, hop_length=hop_length)
    return np.squeeze(rmse)

def mfcc(data, sr, frame_length=2048, hop_length=512, flatten=True):
    mfcc_result = librosa.feature.mfcc(y=data, sr=sr, n_fft=frame_length, hop_length=hop_length)
    return np.squeeze(mfcc_result.T) if not flatten else np.ravel(mfcc_result.T)

def extract_features(data, sr=22050, frame_length=2048, hop_length=512):
    result = np.array([])
    result = np.hstack((
        result, 
        zcr(data, frame_length, hop_length),
        rmse(data, frame_length, hop_length),
        mfcc(data, sr, frame_length, hop_length)
    ))
    return result

def get_predict_feat(path, expected_shape=(1, 2376)):
    d, s_rate = librosa.load(path, duration=2.5, offset=0.6)
    res = extract_features(d)
    # Ensure res is reshaped or padded to match the expected shape
    if res.shape != expected_shape:
        flat_size = np.prod(expected_shape)
        if res.size < flat_size:
            # Pad if the size is smaller than expected
            pad_width = (0, flat_size - res.size)
            res = np.pad(res, pad_width=pad_width, mode='constant')
        else:
            # Resize if the size is larger than expected
            res = np.resize(res, expected_shape)
    i_result = scaler.transform(res.reshape(1, -1))
    final_result = np.expand_dims(i_result, axis=2)
    return final_result

# Add your prediction function
def prediction(path1, predicted_emo=[]):
    print(path1)
    res = get_predict_feat(path1)
    predictions = loaded_model.predict(res)
    # Get the index of the label with the highest confidence score
    predicted_label_index = np.argmax(predictions)
    # List to store confidence scores
    confidence_scores = []
    # Display predicted emotion and confidence for each label
    print(f"\nPredicted Emotion: {label_names[predicted_label_index]}")
    predicted_emo.append(label_names[predicted_label_index])
    for label_index, label_name in enumerate(label_names):
        confidence_score = predictions[0][label_index]
        confidence_score = 0 if confidence_score < 0.001 else confidence_score
        confidence_scores.append({'label': label_name, 'confidence': confidence_score})
    print("\n")
    sorted_confidence_scores = sorted(confidence_scores, key=lambda x: x['confidence'], reverse=True)
    return sorted_confidence_scores

2025-05-18 00:46:47.786718: E external/local_xla/xla/stream_executor/cuda/cuda_platform.cc:51] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: CUDA_ERROR_NO_DEVICE: no CUDA-capable device is detected
2025-05-18 00:46:47.787406: I external/local_xla/xla/stream_executor/cuda/cuda_diagnostics.cc:167] env: CUDA_VISIBLE_DEVICES="-1"
2025-05-18 00:46:47.787814: I external/local_xla/xla/stream_executor/cuda/cuda_diagnostics.cc:170] CUDA_VISIBLE_DEVICES is set to -1 - this hides all GPUs from CUDA
2025-05-18 00:46:47.787817: I external/local_xla/xla/stream_executor/cuda/cuda_diagnostics.cc:178] verbose logging is disabled. Rerun with verbose logging (usually --v=1 or --vmodule=cuda_diagnostics=1) to get more diagnostic output from this module
2025-05-18 00:46:47.787819: I external/local_xla/xla/stream_executor/cuda/cuda_diagnostics.cc:183] retrieving CUDA diagnostic information for host: aegis-Thin-GF63-12UCX
2025-05-18 00:46:47.787824: I external/local_xla/xla/stream_execu

Available emotions: ['angry', 'disgust', 'fear', 'happy', 'neutral', 'sad', 'surprise']


/home/aegis/Research/Machine Learning in HEP/mlenv/lib/python3.12/site-packages/sklearn/base.py:376: InconsistentVersionWarning: Trying to unpickle estimator StandardScaler from version 1.2.2 when using version 1.5.2. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/home/aegis/Research/Machine Learning in HEP/mlenv/lib/python3.12/site-packages/sklearn/base.py:376: InconsistentVersionWarning: Trying to unpickle estimator OneHotEncoder from version 1.2.2 when using version 1.5.2. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


In [9]:
for emo in label_names[:-1]:
    prediction(f"emotion_audio/{emo}_synthetic.wav")

emotion_audio/angry_synthetic.wav
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 335ms/step

Predicted Emotion: angry


emotion_audio/disgust_synthetic.wav
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 113ms/step

Predicted Emotion: angry


emotion_audio/fear_synthetic.wav
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 109ms/step

Predicted Emotion: angry


emotion_audio/happy_synthetic.wav
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 96ms/step

Predicted Emotion: disgust


emotion_audio/neutral_synthetic.wav
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 96ms/step

Predicted Emotion: fear


emotion_audio/sad_synthetic.wav
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 106ms/step

Predicted Emotion: neutral




In [24]:
import numpy as np
import librosa
import tensorflow as tf
import pickle

# === Load pre-trained model and preprocessing tools ===
with open('results/CNN_model.json', 'r') as json_file:
    loaded_model_json = json_file.read()
loaded_model = tf.keras.models.model_from_json(loaded_model_json)
loaded_model.load_weights("results/best_model.weights.h5")
loaded_model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

with open("results/scaler.pickle", "rb") as f:
    scaler = pickle.load(f)
with open("results/encoder.pickle", "rb") as f:
    encoder = pickle.load(f)

label_names = list(encoder.categories_[0])
print("Available emotions:", label_names)

# === Feature extraction ===
def zcr(data, frame_length=2048, hop_length=512):
    return np.squeeze(librosa.feature.zero_crossing_rate(data, frame_length=frame_length, hop_length=hop_length))

def rmse(data, frame_length=2048, hop_length=512):
    return np.squeeze(librosa.feature.rms(y=data, frame_length=frame_length, hop_length=hop_length))

def mfcc(data, sr, frame_length=2048, hop_length=512):
    return np.ravel(librosa.feature.mfcc(y=data, sr=sr, n_fft=frame_length, hop_length=hop_length).T)

def extract_features(data, sr=22050):
    z = zcr(data)
    r = rmse(data)
    m = mfcc(data, sr)
    return np.concatenate((z, r, m), axis=0)

# === Learn to shape features ===
class FeatureShaper(tf.Module):
    def __init__(self, init_vector):
        super().__init__()
        self.trainable_features = tf.Variable(init_vector, dtype=tf.float32, trainable=True)

    def __call__(self):
        return self.trainable_features

def save_audio_from_shaped_features(shaped_vector, save_path, sr=22050, frame_length=2048, hop_length=512):
    shaped_vector = shaped_vector.numpy().flatten() if isinstance(shaped_vector, tf.Tensor) else shaped_vector

    # Fixed calculation
    n_frames = shaped_vector.shape[0] // 22  # Each frame has 22 features: 1 ZCR, 1 RMSE, 20 MFCCs
    print(shaped_vector.shape[0])

    mfcc_flat = shaped_vector[2 * n_frames:]  # Skip ZCR + RMSE
    if mfcc_flat.size != 20 * n_frames:
        raise ValueError(f"Expected {20 * n_frames} MFCC values, got {mfcc_flat.size}")

    mfcc = mfcc_flat.reshape((n_frames, 20)).T  # shape = [20, n_frames]

    # Invert MFCC to audio
    audio = librosa.feature.inverse.mfcc_to_audio(mfcc, sr=sr, n_fft=frame_length, hop_length=hop_length)

    # Save audio
    sf.write(save_path, audio, sr)
    print(f"Saved audio to: {save_path}")
# === Main optimization ===
def generate_shaped_feature(epochs=500, target_emotion='happy', learning_rate=0.01):
    sr = 22050
    duration = 2.5
    samples = int(sr * duration)

    # Step 1: Generate random noise and extract features
    noise = np.random.randn(samples)
    base_features = extract_features(noise, sr=sr)

    # Step 2: Make sure features match expected shape
    if base_features.shape[0] != 2376:
        base_features = np.resize(base_features, (2376,))

    # Step 3: Create a trainable feature vector
    shaper = FeatureShaper(init_vector=base_features.copy())
    optimizer = tf.keras.optimizers.Adam(learning_rate=learning_rate)

    # Step 4: Define target emotion index
    if target_emotion not in label_names:
        raise ValueError(f"'{target_emotion}' not found in {label_names}")
    target_index = label_names.index(target_emotion)

    # Step 5: Train loop
    scaler_mean = tf.constant(scaler.mean_, dtype=tf.float32)
    scaler_scale = tf.constant(scaler.scale_, dtype=tf.float32)
    
    for epoch in range(epochs):
        with tf.GradientTape() as tape:
            shaped = shaper()
            shaped_reshaped = tf.reshape(shaped, (1, -1))
            scaled = (shaped_reshaped - scaler_mean) / scaler_scale  # TF version of StandardScaler
            model_input = tf.expand_dims(scaled, axis=2)
            prediction = loaded_model(model_input, training=False)
            loss = -tf.math.log(prediction[0, target_index] + 1e-8)

        grads = tape.gradient(loss, [shaper.trainable_features])
        optimizer.apply_gradients(zip(grads, [shaper.trainable_features]))

        if epoch % 50 == 0 or epoch == epochs - 1:
            pred_label = label_names[np.argmax(prediction)]
            conf = float(prediction[0, target_index])
            print(f"Epoch {epoch}: Loss={loss.numpy():.4f}, Pred={pred_label}, Confidence={conf:.4f}")


    # Final result
    final_scaled = scaler.transform(shaper().numpy().reshape(1, -1))
    final_input = np.expand_dims(final_scaled, axis=2)
    final_pred = loaded_model(final_input)
    final_probs = final_pred.numpy().flatten()
    print("\nFinal Prediction:")
    for i, prob in enumerate(final_probs):
        print(f"{label_names[i]:<10}: {prob:.4f}")
        
    final_vector = shaper().numpy().flatten()
    save_audio_from_shaped_features(final_vector, "shaped_output.wav")

# === Run it ===
generate_shaped_feature(epochs=500, target_emotion='happy')


/home/aegis/Research/Machine Learning in HEP/mlenv/lib/python3.12/site-packages/sklearn/base.py:376: InconsistentVersionWarning: Trying to unpickle estimator StandardScaler from version 1.2.2 when using version 1.5.2. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/home/aegis/Research/Machine Learning in HEP/mlenv/lib/python3.12/site-packages/sklearn/base.py:376: InconsistentVersionWarning: Trying to unpickle estimator OneHotEncoder from version 1.2.2 when using version 1.5.2. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


Available emotions: ['angry', 'disgust', 'fear', 'happy', 'neutral', 'sad', 'surprise']
Epoch 0: Loss=18.3806, Pred=angry, Confidence=0.0000
Epoch 50: Loss=9.5106, Pred=angry, Confidence=0.0001
Epoch 100: Loss=1.7396, Pred=angry, Confidence=0.1756
Epoch 150: Loss=0.0696, Pred=happy, Confidence=0.9327
Epoch 200: Loss=0.0346, Pred=happy, Confidence=0.9660
Epoch 250: Loss=0.0228, Pred=happy, Confidence=0.9774
Epoch 300: Loss=0.0165, Pred=happy, Confidence=0.9836
Epoch 350: Loss=0.0118, Pred=happy, Confidence=0.9882
Epoch 400: Loss=0.0090, Pred=happy, Confidence=0.9910
Epoch 450: Loss=0.0072, Pred=happy, Confidence=0.9928
Epoch 499: Loss=0.0059, Pred=happy, Confidence=0.9941

Final Prediction:
angry     : 0.0027
disgust   : 0.0000
fear      : 0.0017
happy     : 0.9941
neutral   : 0.0008
sad       : 0.0000
surprise  : 0.0008
2376
Saved audio to: shaped_output.wav


In [22]:
def zcr(data, frame_length, hop_length):
    zcr = librosa.feature.zero_crossing_rate(data, frame_length=frame_length, hop_length=hop_length)
    return np.squeeze(zcr)

def rmse(data, frame_length=2048, hop_length=512):
    rmse = librosa.feature.rms(y=data, frame_length=frame_length, hop_length=hop_length)
    return np.squeeze(rmse)

def mfcc(data, sr, frame_length=2048, hop_length=512, flatten=True):
    mfcc_result = librosa.feature.mfcc(y=data, sr=sr, n_fft=frame_length, hop_length=hop_length)
    return np.squeeze(mfcc_result.T) if not flatten else np.ravel(mfcc_result.T)

def extract_features(data, sr=22050, frame_length=2048, hop_length=512):
    result = np.array([])
    result = np.hstack((
        result, 
        zcr(data, frame_length, hop_length),
        rmse(data, frame_length, hop_length),
        mfcc(data, sr, frame_length, hop_length)
    ))
    return result

def get_predict_feat(path, expected_shape=(1, 2376)):
    d, s_rate = librosa.load(path, duration=2.5, offset=0.6)
    res = extract_features(d)
    # Ensure res is reshaped or padded to match the expected shape
    if res.shape != expected_shape:
        flat_size = np.prod(expected_shape)
        if res.size < flat_size:
            # Pad if the size is smaller than expected
            pad_width = (0, flat_size - res.size)
            res = np.pad(res, pad_width=pad_width, mode='constant')
        else:
            # Resize if the size is larger than expected
            res = np.resize(res, expected_shape)
    i_result = scaler.transform(res.reshape(1, -1))
    final_result = np.expand_dims(i_result, axis=2)
    return final_result
def prediction(path1, predicted_emo=[]):
    print(path1)
    res = get_predict_feat(path1)
    predictions = loaded_model.predict(res)
    # Get the index of the label with the highest confidence score
    predicted_label_index = np.argmax(predictions)
    # List to store confidence scores
    confidence_scores = []
    # Display predicted emotion and confidence for each label
    print(f"\nPredicted Emotion: {label_names[predicted_label_index]}")
    predicted_emo.append(label_names[predicted_label_index])
    for label_index, label_name in enumerate(label_names):
        confidence_score = predictions[0][label_index]
        confidence_score = 0 if confidence_score < 0.001 else confidence_score
        confidence_scores.append({'label': label_name, 'confidence': confidence_score})
    print("\n")
    sorted_confidence_scores = sorted(confidence_scores, key=lambda x: x['confidence'], reverse=True)
    return sorted_confidence_scores

In [23]:
prediction("shaped_output.wav")

shaped_output.wav
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 185ms/step

Predicted Emotion: fear




[{'label': 'fear', 'confidence': 0.8865163},
 {'label': 'angry', 'confidence': 0.11304176},
 {'label': 'disgust', 'confidence': 0},
 {'label': 'happy', 'confidence': 0},
 {'label': 'neutral', 'confidence': 0},
 {'label': 'sad', 'confidence': 0},
 {'label': 'surprise', 'confidence': 0}]